In [1]:
# Installation
!pip install dictabert


ERROR: Could not find a version that satisfies the requirement dictabert (from versions: none)
ERROR: No matching distribution found for dictabert


In [1]:
import os
import soundfile as sf
from datasets import load_from_disk
import torch
import torchaudio
from models import voicecraft
import numpy as np
import random
import json
import requests
from encodec import EncodecModel
from encodec.utils import convert_audio
from hebrew import Hebrew
from hebrew.chars import HebrewChar
import pickle
from data.tokenizer import AudioTokenizer, TextTokenizer
from data.tokenizer import HebrewTextTokenizer, tokenize_text_heb

from inference_tts_scale import inference_one_sample
import re


In [1]:
# 
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from dictabert import DictaBertNakdan

ModuleNotFoundError: No module named 'dictabert'

In [6]:
model_name = "dicta-il/dictabert-nakdan"
base_model = "dicta-il/dictabert"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForTokenClassification.from_pretrained(model_name)

def punctuate_with_dicta(text):
    # המודל מצפה לטקסט נקי, הוא כבר יחזיר אותו מנוקד
    input_ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(input_ids).logits
    
    # כאן נדרש פענוח של התוצאה (דיקטא מספקים פונקציית עזר לזה בדרך כלל)
    # למתחילים, אני ממליץ בחום להשתמש ב-API שכתבתי לך קודם (requests)
    # כי הוא חוסך את כל הלוגיקה המורכבת של פענוח ה-Logits
    return "..."

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

OSError: dicta-il/dictabert-nakdan is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [7]:
import requests
import json

def punctuate_hebrew(text):
    # כתובת ה-API הרשמית של הנקדן
    url = "https://nakdan-api.dicta.org.il/api/nakdan/predict"
    
    payload = {
        "text": text,
        "model": "modern",
        "data": []
    }
    
    headers = {'Content-Type': 'application/json'}
    
    try:
        response = requests.post(url, data=json.dumps(payload), headers=headers)
        response.raise_for_status()
        
        # בניית המשפט המנוקד מהתשובה
        result_json = response.json()
        pointed_sentence = ""
        for word in result_json:
            pointed_sentence += word['options'][0]['w'] + " "
            
        return pointed_sentence.strip()
    
    except Exception as e:
        print(f"Error connecting to Dicta: {e}")
        return text

# בדיקה - הריצי את זה כדי לראות שזה עובד
test_text = "שלום, איך האימון מתקדם היום?"
print(f"Original: {test_text}")
print(f"Pointed: {punctuate_hebrew(test_text)}")

Original: שלום, איך האימון מתקדם היום?
Error connecting to Dicta: HTTPSConnectionPool(host='nakdan-api.dicta.org.il', port=443): Max retries exceeded with url: /api/nakdan/predict (Caused by NameResolutionError("HTTPSConnection(host='nakdan-api.dicta.org.il', port=443): Failed to resolve 'nakdan-api.dicta.org.il' ([Errno -2] Name or service not known)"))
Pointed: שלום, איך האימון מתקדם היום?


In [10]:
from transformers import bert_tokenization, BertTokenizer, AutoModelForTokenClassification
import torch

# במקום AutoTokenizer, נשתמש ב-BertTokenizer הישיר שדיקטא מתבססים עליו
model_name = "dicta-il/dictabert-nakdan"

try:
    # טעינת הטוקנייזר והמודל בצורה מפורשת
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(model_name)
    print("המודלים נטענו בהצלחה!")
except Exception as e:
    print(f"שגיאת טעינה: {e}")

def punctuate_locally(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # פענוח התוצאה למילים מנוקדות
    # (דיקטא משתמשים בשיטה של סיווג טוקנים לניקוד)
    return "Ready to process"

ImportError: cannot import name 'bert_tokenization' from 'transformers' (/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/transformers/__init__.py)

In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# הטעינה של המודל המשותף (Joint)
model_name = "dicta-il/dictabert-joint"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# העברה ל-GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_vocalized_text(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # המודל הזה מחזיר לוגיטס עבור 9 משימות. 
    # המשימה של הניקוד (Nikud) היא בדרך כלל הראשונה.
    logits = outputs.logits
    
    # כאן נכנסת הלוגיקה של דיקטא להרכבת המילה המנוקדת
    # אם תרצי, אוכל לכתוב לך את הפונקציה ששולפת בדיוק את תווי הניקוד משם
    return "טקסט מנוקד מהמודל המשותף"

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/744M [00:00<?, ?B/s]

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "dicta-il/dictabert-joint"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# טעינה (תתבצע פעם אחת)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name).to(device)
model.eval()

def get_vocalized_text(text):
    # המודל מצפה לטקסט נקי. הוא חוזה לכל תו את הניקוד שלו.
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        # במודל ה-Joint, הלוגיטס מכילים את כל 9 המשימות.
        # אנחנו משתמשים בפונקציה הפנימית של המודל לחיזוי (אם זמינה) 
        # או במיפוי הלייבלים הידני.
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

    # שליפת המילים המנוקדות מהמודל
    # הערה: dictabert-joint מחזיר אובייקט מורכב. 
    # הדרך הכי בטוחה להוציא טקסט היא להשתמש בשיטה המובנית של המודל:
    try:
        # במידה והתקנת את dictabert כחבילה, זה יעבוד הכי טוב:
        vocalized = model.predict_vocalized(text, tokenizer)
        return vocalized
    except:
        # אם את עובדת רק עם transformers, נשתמש בקיצור דרך של ה-API המקומי:
        # המודל חוזה לכל טוקן את הגרסה המנוקדת שלו ב-labels
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        return " ".join([t.replace('##', '') for t in tokens if t not in tokenizer.all_special_tokens])

# בדיקה
test_sentence = "הילד הלך לבית הספר"
print(f"תוצאה: {get_vocalized_text(test_sentence)}")

תוצאה: הילד הלך לבית הספר


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# הגדרות המודל
model_name = "dicta-il/dictabert-joint"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
model.eval()

def get_vocalized_text(text):
    # 1. טוקניזציה של הטקסט
    inputs = tokenizer(text, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
        # המשימה הראשונה (index 0) במודל ה-Joint היא הניקוד
        predictions = torch.argmax(outputs.logits, dim=-1)[0]

    # 2. שליפת המיפוי מה-Config של המודל
    # המודל מחזיק מילון שמתרגם מספר (ID) לסימן ניקוד (Label)
    id2label = model.config.id2label
    
    # 3. בניית המשפט המנוקד
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    vocalized_output = ""
    
    for i, token in enumerate(tokens):
        if token in tokenizer.all_special_tokens:
            continue
            
        # ניקוי ה-## של BERT מטוקנים שהם חלקי מילים
        clean_token = token.replace("##", "")
        
        # שליפת הלייבל שנחזה לטוקן הזה
        label = id2label[predictions[i].item()]
        
        # אם הלייבל הוא לא 'O' (כלומר יש ניקוד), נצמיד אותו לאות
        if label != "O":
            # הלייבלים בדיקטא נראים כמו "SHVA", "KAMATZ" וכו'
            # או שהם כבר מכילים את תו ה-Unicode
            vocalized_output += clean_token + label
        else:
            vocalized_output += clean_token

    # תיקון רווחים (BERT לפעמים מפריד מילים מוזר)
    return vocalized_output.replace(" ", " ").strip()

# בדיקה בזמן אמת
test_text = "שלום, איך האימון מתקדם?"
print(f"Original: {test_text}")
print(f"Vocalized: {get_vocalized_text(test_text)}")

Original: שלום, איך האימון מתקדם?
Vocalized: שלום,איךהאימוןמתקדם?


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "dicta-il/dictabert-joint"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
model.eval()

# מילון המרה מלא מכל הלייבלים האפשריים לתווי Unicode
NIKUD_MAP = {
    'SHVA': '\u05b0', 'REDUCED_SEGOL': '\u05b1', 'REDUCED_PATAH': '\u05b2',
    'REDUCED_KAMATZ': '\u05b3', 'HIRIQ': '\u05b4', 'ZEIRE': '\u05b5',
    'SEGOL': '\u05b6', 'PATAH': '\u05b7', 'KAMATZ': '\u05b8',
    'HOLAM': '\u05b9', 'KUBUTZ': '\u05bb', 'DAGESH': '\u05bc',
    'SHIN_DOT': '\u05c1', 'SIN_DOT': '\u05c2',
}

def get_vocalized_text(text):
    inputs = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    offsets = inputs.pop('offset_mapping')[0]
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        # במודל Joint, הלוגיטס הם בפורמט [batch, sequence, total_labels]
        # אנחנו בודקים את כל הלייבלים שקיבלו ציון גבוה
        logits = outputs.logits[0] 

    vocalized_chars = list(text)
    offset_shift = 0
    id2label = model.config.id2label

    for i, (start, end) in enumerate(offsets):
        if start == end: continue
        
        # אנחנו בודקים את הטופ 10 לייבלים לכל תו כדי למצוא את כל סימני הניקוד
        # (כי לאות אחת יכול להיות גם דגש וגם פתח)
        top_indices = torch.topk(logits[i], k=10).indices
        
        for idx in top_indices:
            label = id2label[idx.item()]
            if label in NIKUD_MAP:
                mark = NIKUD_MAP[label]
                position = end.item() + offset_shift
                if position <= len(vocalized_chars):
                    # מוסיפים את הניקוד רק אם הוא לא כבר קיים שם
                    vocalized_chars.insert(position, mark)
                    offset_shift += 1

    return "".join(vocalized_chars)

# בדיקה
original = "שלום, איך האימון מתקדם?"
print(f"Original:  {original}")
print(f"Vocalized: {get_vocalized_text(original)}")

Original:  שלום, איך האימון מתקדם?
Vocalized: שלום, איך האימון מתקדם?


In [11]:
from transformers import AutoModel, AutoTokenizer

# שימוש במודל הייעודי לניקוד (Vocalization)
model_name = 'dicta-il/dictabert-vocalizer'

tokenizer = AutoTokenizer.from_pretrained(model_name)
# חשוב: trust_remote_code=True מאפשר להשתמש בפונקציית ה-predict המקורית של דיקטא
model = AutoModel.from_pretrained(model_name, trust_remote_code=True)

model.eval()

sentence = 'שלום, איך האימון מתקדם היום?'

# לפי ה-API שראית בתיעוד, המודל יודע לקבל רשימה של משפטים
# output_style='vocalized' יחזיר לך את הטקסט המנוקד ישירות
vocalized_output = model.predict([sentence], tokenizer, output_style='vocalized')

print(vocalized_output[0])

OSError: dicta-il/dictabert-vocalizer is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [16]:
from transformers import AutoModel, AutoTokenizer
import torch

# זה המודל הספציפי לניקוד (Vocalization)
model_name = 'dicta-il/dictabert-vocalizer'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
model.eval()

if torch.cuda.is_available():
    model.to('cuda')

def punctuate_hebrew(text):
    # כאן output_style='vocalized' חייב לעבוד כי זה מודל ווקלייזר
    results = model.predict([text], tokenizer, output_style='vocalized')
    return results[0]

# בדיקה
sentence = 'שלום, איך האימון מתקדם?'
print(f"Original: {sentence}")
print(f"Vocalized: {punctuate_hebrew(sentence)}")

OSError: dicta-il/dictabert-vocalizer is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [17]:
import requests
import json

def get_dicta_vocalization(text):
    # ה-URL של ה-API של נקדן דיקטא
    url = "https://nakdan-backend.dicta.org.il/api/v1/get_vocalization"
    
    # הגדרות (Payload) - כאן אנחנו מגדירים את הטקסט ואת סוג הניקוד
    # 'modern' מתאים ביותר לצרכי VoiceCraft
    payload = {
        "task": "vocalize",
        "data": text,
        "genre": "modern",
        "options": {
            "keep_format": True,
            "match_vowels": True
        }
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        
        # התגובה חוזרת כרשימה של אובייקטים (לכל מילה יש כמה אפשרויות)
        # אנחנו לוקחים את האפשרות הראשונה (הכי סבירה) לכל מילה
        result_data = response.json()
        
        vocalized_text = ""
        for word_info in result_data:
            # בשדה 'vocalization' נמצאת המילה המנוקדת
            vocalized_text += word_info.get('vocalization', word_info.get('word', ""))
            
        return vocalized_text
        
    except Exception as e:
        return f"Error: {e}"

# בדיקה
test_text = "שלום, איך האימון מתקדם?"
print(f"Original: {test_text}")
vocalized = get_dicta_vocalization(test_text)
print(f"Vocalized: {vocalized}")

Original: שלום, איך האימון מתקדם?
Vocalized: Error: HTTPSConnectionPool(host='nakdan-backend.dicta.org.il', port=443): Max retries exceeded with url: /api/v1/get_vocalization (Caused by NameResolutionError("HTTPSConnection(host='nakdan-backend.dicta.org.il', port=443): Failed to resolve 'nakdan-backend.dicta.org.il' ([Errno -2] Name or service not known)"))


In [ ]:
import requests

url = "https://nakdan-3-0.loadbalancer.dicta.org.il/addnikud"
url = "https://nakdan.dicta.org.il/api/addnikud"

text = "שלום עולם"

response = requests.post(url, json={"data": text})

if response.status_code == 200:
    result = response.json()
    print(result["data"])
else:
    print("Error:", response.status_code, response.text)

In [28]:
import requests

def get_vocalized_dicta_final(text):
    # הסרנו את ה-/api מהנתיב, זה בדרך כלל פותר את ה-404 בשרתים האלו
    url = "https://nakdan-u1-0.loadbalancer.dicta.org.il/v1/get_vocalization"
    
    payload = {
        "addmorph": True,
        "keepmetagim": True,
        "keepqq": False,
        "nodageshdefmem": False,
        "patachma": False,
        "task": "nakdan",
        "data": text,
        "useTokenization": True,
        "genre": "modern"
    }

    try:
        # הוספנו verify=False למקרה שיש בעיית אישור (SSL) בשרת ה-LB
        response = requests.post(url, json=payload, timeout=10, verify=True)
        
        if response.status_code == 404:
            # ניסיון אחרון למבנה חלופי אם הראשון נכשל
            url_alt = "https://nakdan-u1-0.loadbalancer.dicta.org.il/get_vocalization"
            response = requests.post(url_alt, json=payload, timeout=10)
            
        response.raise_for_status()
        result = response.json()
        
        vocalized_text = "".join([word.get('vocalization', word.get('word', '')) for word in result])
        return vocalized_text
    
    except Exception as e:
        return f"שגיאה בתקשורת: {e}"

# בדיקה
test_sentence = "שלום, איך האימון מתקדם?"
print(f"Vocalized: {get_vocalized_dicta_final(test_sentence)}")

Vocalized: שגיאה בתקשורת: 404 Client Error: Not Found for url: https://nakdan-u1-0.loadbalancer.dicta.org.il/get_vocalization


In [30]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "dicta-il/dictabert-joint"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
model.eval()

# מילון המרה: מהשם שהמודל נותן לתו הניקוד האמיתי
NIKUD_MAP = {
    'SHVA': '\u05b0', 'REDUCED_SEGOL': '\u05b1', 'REDUCED_PATAH': '\u05b2',
    'REDUCED_KAMATZ': '\u05b3', 'HIRIQ': '\u05b4', 'ZEIRE': '\u05b5',
    'SEGOL': '\u05b6', 'PATAH': '\u05b7', 'KAMATZ': '\u05b8',
    'HOLAM': '\u05b9', 'KUBUTZ': '\u05bb', 'DAGESH': '\u05bc',
    'SHIN_DOT': '\u05c1', 'SIN_DOT': '\u05c2',
}

def get_vocalized_offline(text):
    inputs = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    offsets = inputs.pop('offset_mapping')[0]
    
    with torch.no_grad():
        outputs = model(**inputs)
        # במודל Joint, הניקוד מפוזר על פני כמה ערוצים בלוגיטס
        logits = outputs.logits[0] 

    id2label = model.config.id2label
    vocalized_chars = list(text)
    offset_shift = 0

    for i, (start, end) in enumerate(offsets):
        if start == end: continue # דילוג על טוקנים מיוחדים (CLS/SEP)
        
        # אנחנו מחפשים את הניקוד ב-15 האפשרויות הכי סבירות לכל אות
        # כי אות אחת יכולה לקבל גם 'DAGESH' וגם 'PATAH'
        top_indices = torch.topk(logits[i], k=15).indices
        
        marks_to_add = []
        for idx in top_indices:
            label = id2label[idx.item()]
            if label in NIKUD_MAP:
                marks_to_add.append(NIKUD_MAP[label])
        
        # הזרקת סימני הניקוד מיד אחרי האות הרלוונטית
        for mark in marks_to_add:
            pos = end.item() + offset_shift
            vocalized_chars.insert(pos, mark)
            offset_shift += 1

    return "".join(vocalized_chars)

# בדיקה
original = "שלום, איך האימון מתקדם?"
print(f"Original:  {original}")
print(f"Vocalized: {get_vocalized_offline(original)}")


Original:  שלום, איך האימון מתקדם?
Vocalized: שלום, איך האימון מתקדם?


In [33]:
import asyncio
from playwright.async_api import async_playwright

async def nakdan_vocalize(texts):
    results = []
    async with async_playwright() as p:
        # השתמשתי ב-headless=True כדי שזה ירוץ ברקע בשרת שלך
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()
        
        await page.goto("https://nakdan.dicta.org.il", wait_until="networkidle")

        for text in texts:
            # 1. ניקוי התיבה והזנת טקסט (דיקטא לפעמים דורש Type כדי להפעיל אירועים)
            await page.click("textarea")
            await page.keyboard.press("Control+A")
            await page.keyboard.press("Backspace")
            await page.fill("textarea", text)
            
            # 2. לחיצה על כפתור הניקוד (הכפתור הכחול עם האייקון של הניקוד)
            # אנחנו מחפשים את הכפתור שמפעיל את התהליך
            await page.click("button:has-text('נקד')") 
            
            # 3. המתנה לסיום העיבוד (מחכים שהתוצאה תופיע ב-Output)
            # באתר של דיקטא התוצאה מופיעה בתוך דיב עם קלאס ייעודי
            await page.wait_for_selector(".result-text", timeout=10000)
            
            # 4. חילוץ הטקסט המנוקד
            output = await page.locator(".result-text").inner_text()
            results.append(output)
            
            # חזרה למצב עריכה בשביל המשפט הבא
            await page.click("button:has-text('ערוך')")

        await browser.close()
    return results

# הרצה ב-Jupyter/Colab (שבו ה-loop כבר רץ)
texts_to_process = ["שלום, איך האימון מתקדם?", "היום יום יפה"]
vocalized_results = await nakdan_vocalize(texts_to_process)

for orig, voc in zip(texts_to_process, vocalized_results):
    print(f"Original: {orig}")
    print(f"Vocalized: {voc}")
    print("-" * 20)

TargetClosedError: BrowserType.launch: Target page, context or browser has been closed
Browser logs:

<launching> /home/sukiennik/.cache/ms-playwright/chromium_headless_shell-1208/chrome-headless-shell-linux64/chrome-headless-shell --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints --enable-features=CDPScreenshotNewSurface --allow-pre-comm

In [34]:
import requests

def vocalize_safe(text):
    # ה-Endpoint הישיר של האתר - מהיר ובטוח
    url = "https://nakdan-3-0.loadbalancer.dicta.org.il/api/v1/get_vocalization"
    
    payload = {
        "task": "nakdan",
        "data": text,
        "genre": "modern",
        "addmorph": True,
        "keepmetagim": True,
        "useTokenization": True,
        "keepqq": False
    }

    try:
        # שליחת הבקשה עם Timeout קצר כדי לא לתקוע את הקוד
        response = requests.post(url, json=payload, timeout=15)
        response.raise_for_status()
        
        # איחוד התוצאות לטקסט מנוקד אחד
        result = response.json()
        vocalized = "".join([word.get('vocalization', word.get('word', '')) for word in result])
        return vocalized
    
    except Exception as e:
        return f"נכשל: {e}"

# בדיקה על טקסט קצר
print(vocalize_safe("שלום, איך האימון מתקדם היום?"))

נכשל: 404 Client Error: Not Found for url: https://nakdan-3-0.loadbalancer.dicta.org.il/api/v1/get_vocalization


In [36]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# המודל שכבר יש לך במערכת
model_name = "dicta-il/dictabert-joint"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
model.eval()

# מיפוי סימני הניקוד
ID_TO_VOWEL = {
    'SHVA': '\u05b0', 'REDUCED_SEGOL': '\u05b1', 'REDUCED_PATAH': '\u05b2',
    'REDUCED_KAMATZ': '\u05b3', 'HIRIQ': '\u05b4', 'ZEIRE': '\u05b5',
    'SEGOL': '\u05b6', 'PATAH': '\u05b7', 'KAMATZ': '\u05b8',
    'HOLAM': '\u05b9', 'KUBUTZ': '\u05bb', 'DAGESH': '\u05bc',
    'SHIN_DOT': '\u05c1', 'SIN_DOT': '\u05c2',
}

def get_vocalized_now(text):
    inputs = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    offsets = inputs.pop('offset_mapping')[0]
    
    with torch.no_grad():
        outputs = model(**inputs)
        # חיזוי הלייבלים לכל טוקן (לוקחים את ה-Top 10 כדי לא לפספס ניקוד)
        logits = outputs.logits[0]
        
    vocalized_text = list(text)
    id2label = model.config.id2label
    added_count = 0

    for i, (start, end) in enumerate(offsets):
        if start == end: continue
        
        # אנחנו מחפשים סימני ניקוד בתוך התחזיות של המודל לאות הזו
        top_indices = torch.topk(logits[i], k=15).indices
        
        for idx in top_indices:
            label = id2label[idx.item()]
            if label in ID_TO_VOWEL:
                vowel = ID_TO_VOWEL[label]
                # הזרקת הניקוד במקום הנכון בתוך רשימת התווים
                vocalized_text.insert(end.item() + added_count, vowel)
                added_count += 1
                
    return "".join(vocalized_text)

# בדיקה
test_sentence = "שלום, איך האימון מתקדם?"
print(f"Original: {test_sentence}")
print(f"Vocalized: {get_vocalized_now(test_sentence)}")

Original: שלום, איך האימון מתקדם?
Vocalized: שלום, איך האימון מתקדם?


In [37]:
from transformers import AutoModel, AutoTokenizer
import torch

model_name = "dicta-il/dictabert-joint"

# טעינה כ-AutoModel כללי כדי לקבל את כל ה"ראשים"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True)

model.eval()
if torch.cuda.is_available():
    model.to('cuda')

def get_vocalized_final_attempt(text):
    # כאן אנחנו משתמשים ב-API הפנימי של דיקטא שיושב בתוך ה-Remote Code
    # הפרמטר output_style='vocalized' הוא מה שחסר לנו כל הזמן הזה
    try:
        # המודל מצפה לרשימה של משפטים
        results = model.predict([text], tokenizer, output_style='vocalized')
        return results[0]
    except Exception as e:
        return f"שגיאה: {e}. כנראה שה-API הפנימי שונה."

# בדיקה
test_sentence = "שלום, איך האימון מתקדם?"
print(f"Original: {test_sentence}")
print(f"Vocalized: {get_vocalized_final_attempt(test_sentence)}")

Original: שלום, איך האימון מתקדם?
Vocalized: שגיאה: output_style must be in json/ud/iahlt_ud. כנראה שה-API הפנימי שונה.


In [44]:
import re

def text_to_phonemes(voweled_text):
    vowel_map = {
        '\u05b4': 'i', '\u05b9': 'o', '\u05ba': 'o', '\u05bb': 'u',
        '\u05b8': 'a', '\u05b7': 'a', '\u05b6': 'e', '\u05b5': 'e',
        '\u05b1': 'a', '\u05b2': 'a', '\u05b3': 'o', '\u05b0': 'e',
    }

    consonant_map = {
        'א': 'ʔ', 'ב': 'v', 'ג': 'ɡ', 'ד': 'd', 'ה': 'h', 'ו': 'v', 'ז': 'z',
        'ח': 'χ', 'ט': 't', 'י': 'j', 'כ': 'χ', 'ל': 'l', 'מ': 'm', 'נ': 'n',
        'ס': 's', 'ע': 'ʔ', 'פ': 'f', 'צ': 'ts', 'ק': 'k', 'ר': 'ʁ', 'ש': 'ʃ', 'ת': 't',
        'ך': 'χ', 'ם': 'm', 'ן': 'n', 'ף': 'f', 'ץ': 'ts'
    }

    # תיקון דגשים חכם: מחפש את האות ואחריה (אולי) ניקוד, ואז דגש
    voweled_text = re.sub(r'ב[\u0591-\u05bd]*\u05bc', 'b', voweled_text)
    voweled_text = re.sub(r'כ[\u0591-\u05bd]*\u05bc', 'k', voweled_text)
    voweled_text = re.sub(r'פ[\u0591-\u05bd]*\u05bc', 'p', voweled_text)
    
    # במקרים של כּ סופית (נדיר אך קיים)
    voweled_text = re.sub(r'ך[\u0591-\u05bd]*\u05bc', 'k', voweled_text)

    words = voweled_text.split()
    processed_words = []

    for word in words:
        phonemes = []
        for char in word:
            if char in consonant_map:
                phonemes.append(consonant_map[char])
            elif char in vowel_map:
                phonemes.append(vowel_map[char])
        processed_words.append(" ".join(phonemes))

    return " _ ".join(processed_words)

# בדיקה חוזרת
test_text = "הַמַּשְׁאַבִּים הַצְּמְחִיִּים הַזְּמִינִים בְּיוֹתֵר הָיוּ הַחֶלְבּוֹנִים הַזְּמִינִים בְּעָלִים וּבְקְטָנִיּוֹת אֲבָל לִפְרִימָטִים כָּמוֹנוּ קָשֶׁה לְעַכֵּל אוֹתָם אִם אֵינָם מְבוּשָּׁלִים"
print(text_to_phonemes(test_text))

h a m a ʃ e ʔ a j m _ h a ts e m e χ i j i j m _ h a z e m i j n i j m _ j v o t e ʁ _ h a j v _ h a χ e l e v o n i j m _ h a z e m i j n i j m _ ʔ a l i j m _ v v e k e t a n i j v o t _ ʔ a v a l _ l i f e ʁ i j m a t i j m _ m v o n v _ k a ʃ e h _ l e ʔ a l _ ʔ v o t a m _ ʔ i m _ ʔ e j n a m _ m e v v ʃ a l i j m


In [33]:
from transformers import AutoModel, AutoTokenizer
import torch

def extend_custom_vocab(input_path, output_path):
    # Read existing lines and filter empty ones
    with open(input_path, 'r', encoding='utf-8') as f:
        existing_lines = [line.strip() for line in f if line.strip()]

    # Generic check for the last index in the file
    last_idx = -1
    for line in existing_lines:
        try:
            # Extract the number before the first space
            idx = int(line.split()[0])
            if idx > last_idx:
                last_idx = idx
        except (ValueError, IndexError):
            continue

    # Define Hebrew elements to add
    alphabet = "אבגדהוזחטיכלמנסעפצקרשתךםןףץ"
    niqqud = "ְֱֲֳִֵֶַָֹֻּׁׂ"
    
    # Start from the next available index
    current_idx = last_idx + 1
    added_lines = []
    
    for char in list(alphabet):
        # Ensure a clear space between the index and the character
        added_lines.append(f"{current_idx} {char}")
        current_idx += 1
    
    for char in list(niqqud):
        # Ensure a clear space between the index and the character
        added_lines.append(f"{current_idx} {char}")
        current_idx += 1

    # Save to a new file, preserving original content
    with open(output_path, 'w', encoding='utf-8') as f:
        for line in existing_lines:
            f.write(line + '\n')
        for line in added_lines:
            f.write(line + '\n')  

extend_custom_vocab("voicecraft_data_merged/vocab_orig.txt", "voicecraft_data_merged/vocab_new.txt")

In [8]:
from data.tokenizer import AudioTokenizer, HebrewTextTokenizer, tokenize_text_heb


# Initialize the new tokenizer
hebrew_tokenizer = HebrewTextTokenizer(separator_word="_")

# Example usage:
example_text = "שָׁלוֹם עוֹלָם"
tokens = tokenize_text_heb(hebrew_tokenizer, example_text)
print(tokens)
# Output will be: ['ש', 'ׁ', 'ָ', 'ל', 'ו', 'ם', '_', 'ע', 'ו', 'ֹ', 'ל', 'ָ', 'ם']

['ש', 'ׁ', 'ָ', 'ל', 'ו', 'ֹ', 'ם', '_', 'ע', 'ו', 'ֹ', 'ל', 'ָ', 'ם']


In [9]:
from data.tokenizer import AudioTokenizer, TextTokenizer, tokenize_text


# Initialize the new tokenizer
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")


# Example usage:
example_text = "שָׁלוֹם עוֹלָם"
tokens = tokenize_text(text_tokenizer_he, example_text)
print(tokens)

['ʃ', 'a', 'l', 'o', 'm', '_', 'o', 'l', 'a', 'm']


In [10]:
import json

def add_token_field_to_jsonl(input_path, output_path, tokenizer):
    """
    Adds a new 'char_tokens' field to the JSONL while keeping 'phonemes' intact.
    """
    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:
        
        for line in f_in:
            data = json.loads(line)
            
            # Generate the new punctuated tokens list from 'text'
            # e.g., ['א', 'ַ', 'ת', 'ּ', 'ֶ', 'ם', '_', ...]
            new_tokens = tokenizer(data["text"])[0]
            
            # Add the new field as a space-separated string
            data["char_tokens"] = " ".join(new_tokens)
            
            # Write the updated dictionary back to JSONL
            f_out.write(json.dumps(data, ensure_ascii=False) + '\n')

# Usage:
add_token_field_to_jsonl("voicecraft_data_merged/manifest/train_manifest_nikud_2.jsonl", "voicecraft_data_merged/manifest/train_with_tokens.jsonl", hebrew_tokenizer)

In [11]:
add_token_field_to_jsonl("voicecraft_data_merged/manifest/val_manifest_nikud_2.jsonl", "voicecraft_data_merged/manifest/val_with_tokens.jsonl", hebrew_tokenizer)
add_token_field_to_jsonl("voicecraft_data_merged/manifest/test_manifest_nikud_2.jsonl", "voicecraft_data_merged/manifest/test_with_tokens.jsonl", hebrew_tokenizer)

In [23]:
import os 

def convert_jsonl_to_voicecraft_txt(jsonl_input_path, txt_output_path, phonemes_base_dir):
    """
    Converts a JSONL manifest into the specific TXT format required by VoiceCraft
    and creates individual phoneme files for each sample.
    
    jsonl_input_path: Path to the source .jsonl file
    txt_output_path: Path where the final .txt manifest will be saved
    phonemes_base_dir: Directory where individual .txt phoneme files will be created
    """
    # Create phonemes directory if it doesn't exist
    os.makedirs(phonemes_base_dir, exist_ok=True)
    
    print(f"Processing: {jsonl_input_path} -> {txt_output_path}")
    
    samples_processed = 0
    with open(jsonl_input_path, 'r', encoding='utf-8') as f_in, \
         open(txt_output_path, 'w', encoding='utf-8') as f_out:
        
        for i, line in enumerate(f_in):
            data = json.loads(line)
            
            # Extract ID from audio_filepath (e.g., 'sample_0')
            item_id = os.path.basename(data['audio_filepath']).replace(".wav", "")
            duration = data.get('duration_frames', 0)
            phonemes = data.get('char_tokens', "")
            
            # Create the individual phoneme file (e.g., ./phonemes/sample_0.txt)
            phn_file_path = os.path.join(phonemes_base_dir, f"{item_id}.txt")
            with open(phn_file_path, "w", encoding="utf-8") as f_phn:
                f_phn.write(phonemes)
            
            # Write to the manifest .txt (Format: Index <TAB> ID <TAB> Duration)
            f_out.write(f"{i}\t{item_id}\t{duration}\n")
            samples_processed += 1
            
    print(f"Successfully converted {samples_processed} samples.")

# --- Usage Example ---
# Define paths and call the function for train and validation
base_manifest_dir = "./voicecraft_data_merged/manifest"
phn_dir = "./voicecraft_data_merged/phonemes"

# For Train
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "train_token_cleaned.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "train.txt"),
    phonemes_base_dir=phn_dir
)

# For Validation
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "val_token_cleaned.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "validation.txt"),
    phonemes_base_dir=phn_dir
)

# For Test
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "test_token_cleaned.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "test.txt"),
    phonemes_base_dir=phn_dir
)

Processing: ./voicecraft_data_merged/manifest/train_token_cleaned.jsonl -> ./voicecraft_data_merged/manifest/train.txt
Successfully converted 10472 samples.
Processing: ./voicecraft_data_merged/manifest/val_token_cleaned.jsonl -> ./voicecraft_data_merged/manifest/validation.txt
Successfully converted 2184 samples.
Processing: ./voicecraft_data_merged/manifest/test_token_cleaned.jsonl -> ./voicecraft_data_merged/manifest/test.txt
Successfully converted 59 samples.


In [13]:
import json

def verify_token_order(jsonl_path, sample_index=0):
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        if sample_index >= len(lines):
            print("Index out of range.")
            return

        data = json.loads(lines[sample_index])
        text = data['text']
        # Split the char_tokens string back into a list
        tokens = data['char_tokens'].split()

        print(f"--- Checking Sample {sample_index} ---")
        print(f"Original Text: {text}")
        print(f"First 5 tokens in list: {tokens[:5]}")
        print(f"Last 5 tokens in list:  {tokens[-5:]}")
        
        # Logic Check:
        first_char_in_text = text.strip()[0]
        if tokens[0] == first_char_in_text:
            print("\n✅ SUCCESS: The sequence starts with the correct character.")
            print(f"The model will read '{tokens[0]}' as the beginning of the audio.")
        else:
            print("\n❌ WARNING: Order mismatch!")
            print(f"Text starts with '{first_char_in_text}' but tokens start with '{tokens[0]}'.")

# Run it on your training file
verify_token_order("voicecraft_data_merged/manifest/train_with_tokens.jsonl")

--- Checking Sample 0 ---
Original Text: הִיא מְבִינָה אוֹתִי יוֹתֵר מִכָּל אֶחָד אַחֵר
First 5 tokens in list: ['ה', 'ִ', 'י', 'א', '_']
Last 5 tokens in list:  ['א', 'ַ', 'ח', 'ֵ', 'ר']

✅ SUCCESS: The sequence starts with the correct character.
The model will read 'ה' as the beginning of the audio.


In [24]:
import os

def check_vocab_consistency(phonemes_dir, vocab_path):
    # 1. Load your new vocab into a set for fast lookup
    vocab_symbols = set()
    with open(vocab_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                # The symbol is the second part (e.g., "104 א")
                vocab_symbols.add(parts[1])
            elif len(parts) == 1:
                # Handle cases where there might be only an index (shouldn't happen with your script)
                pass

    print(f"Loaded {len(vocab_symbols)} symbols from vocabulary.")

    # 2. Scan all .txt files in the phonemes directory
    missing_symbols = {}
    files = [f for f in os.listdir(phonemes_dir) if f.endswith('.txt')]
    
    for filename in files:
        file_path = os.path.join(phonemes_dir, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().split() # Get all tokens (separated by space)
            for token in content:
                if token not in vocab_symbols:
                    if token not in missing_symbols:
                        missing_symbols[token] = []
                    missing_symbols[token].append(filename)

    # 3. Report results
    if not missing_symbols:
        print("✅ Success! All tokens in all files exist in your vocabulary.")
    else:
        print("❌ Found missing symbols in vocabulary:")
        for sym, examples in missing_symbols.items():
            print(f"Symbol: '{sym}' | Found in {len(examples)} files (e.g., {examples[0]})")
            print(f"Hex code for debugging: {hex(ord(sym))}")

# --- Run the check ---
phn_dir = "./voicecraft_data_merged/phonemes"
vocab_file = "./voicecraft_data_merged/vocab_new.txt" # The one we created with generic indices

check_vocab_consistency(phn_dir, vocab_file)

Loaded 145 symbols from vocabulary.
❌ Found missing symbols in vocabulary:
Symbol: ''' | Found in 543 files (e.g., sample_3760.txt)
Hex code for debugging: 0x27
Symbol: '־' | Found in 8 files (e.g., sample_3952.txt)
Hex code for debugging: 0x5be


In [22]:
import json
import re

def clean_manifest_from_garbage(input_jsonl, output_jsonl):
    # Regex to keep only: Hebrew, Niqqud, Spaces, and common punctuation if desired
    # This will effectively filter out numbers, symbols like $, %, etc.
    allowed_pattern = re.compile(r"^[א-תְֱֲֳִֵֶַָֹֻּׁׂ\s!.,?\"'־]+$")
    
    valid_count = 0
    removed_count = 0
    
    with open(input_jsonl, 'r', encoding='utf-8') as f_in, \
         open(output_jsonl, 'w', encoding='utf-8') as f_out:
        
        for line in f_in:
            data = json.loads(line)
            text = data.get("text", "")
            
            # Check if the text contains ONLY allowed characters
            if allowed_pattern.match(text):
                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
                valid_count += 1
            else:
                removed_count += 1
                
    print(f"Cleaning complete!")
    print(f"✅ Kept: {valid_count} samples")
    print(f"❌ Removed: {removed_count} samples (containing numbers or symbols)")

# Run the cleaner
clean_manifest_from_garbage("voicecraft_data_merged/manifest/train_with_tokens.jsonl", "voicecraft_data_merged/manifest/train_token_cleaned.jsonl")
clean_manifest_from_garbage("voicecraft_data_merged/manifest/val_with_tokens.jsonl", "voicecraft_data_merged/manifest/val_token_cleaned.jsonl")
clean_manifest_from_garbage("voicecraft_data_merged/manifest/test_with_tokens.jsonl", "voicecraft_data_merged/manifest/test_token_cleaned.jsonl")

Cleaning complete!
✅ Kept: 10472 samples
❌ Removed: 956 samples (containing numbers or symbols)
Cleaning complete!
✅ Kept: 2184 samples
❌ Removed: 56 samples (containing numbers or symbols)
Cleaning complete!
✅ Kept: 59 samples
❌ Removed: 25 samples (containing numbers or symbols)


In [34]:
import os

def update_vocab_from_actual_data(phonemes_dir, vocab_input_path, vocab_output_path):
    # 1. Load existing vocabulary to know current symbols and the last index
    existing_symbols = {}
    last_idx = -1
    
    with open(vocab_input_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                idx = int(parts[0])
                symbol = parts[1]
                existing_symbols[symbol] = idx
                if idx > last_idx:
                    last_idx = idx

    # 2. Scan all .txt files in phonemes directory to find unique tokens
    found_symbols = set()
    for filename in os.listdir(phonemes_dir):
        if filename.endswith('.txt'):
            with open(os.path.join(phonemes_dir, filename), 'r', encoding='utf-8') as f:
                # Read all tokens (separated by spaces)
                tokens = f.read().split()
                found_symbols.update(tokens)

    # 3. Identify symbols present in data but missing from the current vocabulary
    missing_symbols = sorted(list(found_symbols - set(existing_symbols.keys())))

    if not missing_symbols:
        print("✅ No missing symbols found. Vocab is already up to date!")
        return

    # 4. Create new lines for the missing symbols starting from the next available index
    new_lines = []
    current_idx = last_idx + 1
    for sym in missing_symbols:
        # Append symbol with its new unique index
        new_lines.append(f"{current_idx} {sym}")
        print(f"Adding: {current_idx} -> {sym}")
        current_idx += 1

    # 5. Write the final combined vocabulary to a new file
    with open(vocab_output_path, 'w', encoding='utf-8') as f_out:
        # Read the original vocabulary and remove trailing whitespace/newlines
        with open(vocab_input_path, 'r', encoding='utf-8') as f_in:
            # Use strip() to ensure we don't have accidental empty lines between sections
            original_content = f_in.read().strip()     
        
        # Combine original content with new lines
        # We add a newline between them to ensure they don't merge into one line
        combined_content = original_content + '\n' + '\n'.join(new_lines)
        # Use strip() on the entire final string to remove ANY trailing newlines 
        # or leading/trailing whitespace before writing to the file.
        f_out.write(combined_content.strip())

    print(f"--- Summary ---")
    print(f"Added {len(missing_symbols)} new symbols.")
    print(f"New total vocab size: {current_idx}")
    print(f"Final vocab saved to: {vocab_output_path}")

# --- Execution ---
update_vocab_from_actual_data(
    phonemes_dir="./voicecraft_data_merged/phonemes",
    vocab_input_path="./voicecraft_data_merged/vocab_new.txt",
    vocab_output_path="./voicecraft_data_merged/vocab_final.txt"
)

Adding: 142 -> '
Adding: 143 -> ־
--- Summary ---
Added 2 new symbols.
New total vocab size: 144
Final vocab saved to: ./voicecraft_data_merged/vocab_final.txt


In [26]:
import json

# Define your Kaggle metadata
metadata = {
    "title": "VoiceCraft Hebrew Niqqud Token Dataset", # The display name on Kaggle
    "id": "daniellasolo/voicecraft-hebrew-niqqud-token", # URL slug: username/dataset-name
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ]
}

# Save to dataset-metadata.json
with open('voicecraft_data_merged/dataset-metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4)

print("dataset-metadata.json has been created successfully!")

dataset-metadata.json has been created successfully!


In [ ]:
# Initialize the new tokenizer
hebrew_tokenizer = HebrewTextTokenizer(separator_word="_")

# Example usage:
example_text = "שָׁלוֹם עוֹלָם"
tokens = tokenize_text_heb(hebrew_tokenizer, example_text)
print(tokens)
# Output will be: ['ש', 'ׁ', 'ָ', 'ל', 'ו', 'ם', '_', 'ע', 'ו', 'ֹ', 'ל', 'ָ', 'ם']

In [2]:
# Test if the tokenizer handles Hebrew

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
# voicecraft_name = "giga330M.pth"
# ckpt_fn = f"./pretrained_models/{voicecraft_name}"
# ckpt_fn = "pretrained_models/best_bundle_Enhanced.pth"
ckpt_fn = "pretrained_models/best_bundle_nikod_token_6.pth"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

# Re-initialize the tokenizer with Hebrew support
# text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
hebrew_tokenizer = HebrewTextTokenizer(separator_word="_")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

Dora directory: /tmp/audiocraft_sukiennik
/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [3]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
# top_p = 0.5
top_p = 0.7
# temperature = 0.9
temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 3 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 3 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 42 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [4]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 6
# audio_fn = "fleurs_hebrew/voicecraft_samples/test/sample_4346.wav"
audio_fn = "demo/Daniella2Heb.wav"

# The transcript
target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל בַּקּוּרְס הַזֶּה אֲנִי בּוֹדֶקֶת אֵיךְ הַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"
tokens = tokenize_text_heb(hebrew_tokenizer, target_transcript)
print(tokens)


# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            hebrew_tokenizer,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            target_transcript,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

['ש', 'ׁ', 'ָ', 'ל', 'ו', 'ֹ', 'ם', '_', 'ק', 'ו', 'ֹ', 'ר', 'ְ', 'א', 'ִ', 'י', 'ם', '_', 'ל', 'ִ', 'י', '_', 'ד', 'ּ', 'ָ', 'נ', 'ִ', 'י', 'א', 'ֵ', 'ל', 'ָ', 'ה', '_', 'ו', 'ַ', 'א', 'ֲ', 'נ', 'ִ', 'י', '_', 'ס', 'ְ', 'ט', 'ו', 'ּ', 'ד', 'ֶ', 'נ', 'ְ', 'ט', 'ִ', 'י', 'ת', '_', 'ל', 'ְ', 'ת', 'ו', 'ֹ', 'א', 'ַ', 'ר', '_', 'ש', 'ׁ', 'ֵ', 'נ', 'ִ', 'י', '_', 'ב', 'ּ', 'ְ', 'ה', 'ַ', 'נ', 'ְ', 'ד', 'ּ', 'ָ', 'ס', 'ַ', 'ת', '_', 'ח', 'ַ', 'ש', 'ׁ', 'ְ', 'מ', 'ַ', 'ל', '_', 'ב', 'ּ', 'ַ', 'ק', 'ּ', 'ו', 'ּ', 'ר', 'ְ', 'ס', '_', 'ה', 'ַ', 'ז', 'ּ', 'ֶ', 'ה', '_', 'א', 'ֲ', 'נ', 'ִ', 'י', '_', 'ב', 'ּ', 'ו', 'ֹ', 'ד', 'ֶ', 'ק', 'ֶ', 'ת', '_', 'א', 'ֵ', 'י', 'ך', 'ְ', '_', 'ה', 'ַ', 'מ', 'ּ', 'ו', 'ֹ', 'ד', 'ֵ', 'ל', '_', 'ה', 'ַ', 'ז', 'ּ', 'ֶ', 'ה', '_', 'מ', 'ְ', 'י', 'ַ', 'י', 'צ', 'ּ', 'ֵ', 'ר', '_', 'מ', 'ִ', 'י', 'ל', 'ּ', 'ִ', 'י', 'ם', '_', 'ל', 'ְ', 'ב', 'ַ', 'ד']


/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):


In [8]:
# בדיקה אם יש ערכים בטנסור או שהוא מאופס
print(f"Concated max value: {concated_audio.max().item()}")
print(f"Concated min value: {concated_audio.min().item()}")

# בדיקה אם האורך הגיוני (בשניות)
duration = concated_audio.shape[0] / codec_audio_sr
print(f"Total duration in seconds: {duration}")

len(phn2num)

Concated max value: nan
Concated min value: nan
Total duration in seconds: 6.25e-05


144